# 1-1. Transformer와 Attention

## 학습 목표
- Scaled Dot-Product Attention을 NumPy로 스크래치 구현한다.
- Multi-Head Attention으로 확장하고, MHA / MQA / GQA의 차이를 설명한다.
- Positional Encoding (Sinusoidal / RoPE / ALiBi) 의 동기와 수식을 이해한다.
- KV Cache가 추론 속도를 높이는 원리를 설명한다.
- HuggingFace `bert-base-multilingual-cased` 를 이용해 한국어 문장의 attention을 시각화한다.

## 핵심 키워드
`Scaled Dot-Product Attention`, `Causal Mask`, `Multi-Head`, `MQA/GQA`, `Sinusoidal/RoPE/ALiBi`, `KV Cache`

## 먼저 5줄 요약
1. **Transformer**는 "어떤 단어가 문장 속 다른 어떤 단어를 참조해야 할까?"를 학습하는 구조입니다.
2. 그 참조 관계를 계산하는 핵심 블록이 **Attention**입니다.
3. Attention은 "**쿼리**(Q, 내가 찾는 정보), **키**(K, 각 단어의 태그), **값**(V, 각 단어의 내용)"의 3단 검색과 같습니다.
4. GPT 같은 최신 LLM은 이 블록을 수십 번 쌓고, 그 위에 **위치 정보**와 **미래 가리기(causal mask)**를 얹은 구조입니다.
5. 추론 속도의 비밀인 **KV Cache**는 "이미 계산한 키·값은 저장해 두고 재사용"하는 단순한 트릭입니다.

> 💡 이 노트북은 NumPy로 한 줄씩 직접 구현하면서 수식의 의미를 손에 익히는 것이 목표입니다. PyTorch·FlashAttention 등 실전 구현은 Day 2 이후 등장합니다.

## 0. 준비 — "Attention"을 한 줄로 비유하면?

Transformer는 2017년 Vaswani 외, [Attention Is All You Need](https://arxiv.org/abs/1706.03762) 논문에서 등장했다. 오늘 대부분의 LLM (GPT, Llama, Qwen, Mistral)은 이 구조의 디코더(Decoder-only)를 기반으로 한다.

핵심은 **Self-Attention** — 입력 시퀀스 내에서 각 토큰이 다른 토큰을 얼마나 "참조"할지 계산하는 메커니즘이다.

### 📚 도서관 비유로 이해하기

> 당신이 도서관 사서에게 **"전자금융 관련 책 추천해 주세요"** 라고 물었다고 하자.
>
> 1. **Query (Q)** — 당신이 건네는 질문 = "전자금융 관련 책"
> 2. **Key (K)** — 각 책 표지에 붙은 태그 = "법률", "소설", "금융", "프로그래밍"
> 3. **Value (V)** — 책의 실제 내용 (요약·목차)
>
> 사서는 **Q와 모든 K의 관련도**를 계산해 → 관련도가 높은 책일수록 많이 읽어 당신에게 요약해 준다. 이 "가중 평균"이 Attention의 출력이다.

이 비유를 수식으로 쓰면 아래 수식 한 줄이 된다. 0.1절에서 그 수식을 한 조각씩 해체해 본다.

### 이 노트북의 순서
| 절 | 내용 | 학습 포인트 |
|---|---|---|
| 1 | Scaled Dot-Product Attention 직접 구현 | Q·K·V·softmax·scaling의 의미 |
| 2 | Multi-Head로 확장 & MHA/MQA/GQA 비교 | 왜 head가 여러 개? |
| 3 | Positional Encoding | 순서 정보를 어떻게 주입? |
| 4 | KV Cache | 생성이 빠른 이유 |
| 5 | 실제 BERT로 한국어 attention 시각화 | 학습된 모델이 보는 관계 |

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
from common import report_korean_font_status

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)
print('NumPy version:', np.__version__)

# 그래프에 한글 라벨을 그리기 전에 폰트를 찾아 설정
# 한글 폰트가 없으면 "t0, t1, ..." 같은 인덱스 라벨로 대체해 □ 깨짐 방지
print(report_korean_font_status())

## 1. Scaled Dot-Product Attention

수식:

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

- $Q \in \mathbb{R}^{L_q \times d_k}$: query — "내가 뭐를 찾고 있는가"
- $K \in \mathbb{R}^{L_k \times d_k}$: key — "나는 무엇에 대한 정보인가"
- $V \in \mathbb{R}^{L_k \times d_v}$: value — "실제로 전달할 내용"

### 🧩 수식을 4단계로 해체해 읽기

> 문장 "고객은 은행에 예금을 맡겼다"를 가지고, "**맡겼다**"라는 토큰이 다른 토큰에 주는 attention을 계산한다고 해보자.

| 단계 | 수식 조각 | 무엇을 하는가? | 직관 |
|---|---|---|---|
| ① 비교 | $Q K^\top$ | 내 Query와 각 단어의 Key를 내적 | "맡겼다"가 "고객", "은행", "예금" 중 어느 쪽과 얼마나 관련 있는가를 **숫자(score)**로 뽑음 |
| ② 스케일링 | $\div \sqrt{d_k}$ | 분산이 커지지 않게 나눔 | 숫자가 너무 커지면 다음 softmax가 한 값만 1, 나머지 0으로 쏠림 → 학습 불안정 |
| ③ 정규화 | $\text{softmax}(\cdot)$ | 점수를 **0~1 사이의 가중치**로 | 모든 점수의 합이 1이 되도록 → "주의를 100% 나눠 쓴다"는 해석 |
| ④ 가중합 | $\cdot V$ | 가중치로 Value들의 **가중 평균** | 관련도가 높은 토큰의 내용은 많이, 낮은 토큰은 조금 섞어 최종 표현을 만듦 |

### 왜 "Self-"Attention인가?
Q·K·V가 **모두 같은 입력 시퀀스**에서 선형 변환으로 만들어지기 때문이다 (뒤 절 MHA에서 `Wq/Wk/Wv` 행렬로 등장). 즉, 문장이 스스로를 돌아보면서 "이 토큰은 어느 토큰을 얼마나 볼까?"를 결정한다.

### $\sqrt{d_k}$ 로 나누는 이유 (조금 더 자세히)
$Q$와 $K$의 각 원소가 평균 0·분산 1이라면, $Q \cdot K^\top$의 한 원소는 $d_k$개 항의 합 → 분산이 약 $d_k$로 커진다. $\sqrt{d_k}$로 나누면 다시 분산 1로 돌아와서 softmax의 기울기가 살아난다.

In [ ]:
def softmax(x, axis=-1):
    """수치 안정 softmax. 각 행의 최댓값을 먼저 빼서 exp 오버플로우 방지."""
    x_max = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - x_max)
    return e / np.sum(e, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """NumPy로 구현한 Scaled Dot-Product Attention.

    Args:
        Q: (Lq, d_k)  — 내가 찾는 정보
        K: (Lk, d_k)  — 각 토큰의 태그
        V: (Lk, d_v)  — 각 토큰의 실제 내용
        mask: (Lq, Lk) 또는 None. 1 이면 허용, 0 이면 -inf 로 마스킹.

    Returns:
        output: (Lq, d_v)      — attention 결과 (가중 평균된 V)
        attn_weights: (Lq, Lk) — 각 쿼리가 각 키에 준 가중치 (행 합=1)
    """
    d_k = Q.shape[-1]

    # ① 비교: Q와 모든 K의 내적 점수
    scores = Q @ K.T / np.sqrt(d_k)  # (Lq, Lk)  ← ② 스케일링 포함

    # (선택) 미래 토큰 가리기 등: 허용되지 않는 위치는 -무한대로 밀어 softmax 후 0이 되게
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # ③ 정규화: 각 쿼리별로 가중치 합=1
    attn_weights = softmax(scores, axis=-1)

    # ④ 가중합: 가중치 × V
    output = attn_weights @ V  # (Lq, d_v)
    return output, attn_weights


# ─────────────────────────────────────────────────────────────────────
# 가장 작은 예제: 토큰 4개, 각 토큰이 8차원 벡터로 표현된 경우
# (실제 GPT-4 급 모델은 d=4096~12288, L=수천~수만이다.)
# ─────────────────────────────────────────────────────────────────────
L, d = 4, 8
Q = np.random.randn(L, d)
K = np.random.randn(L, d)
V = np.random.randn(L, d)

out, attn = scaled_dot_product_attention(Q, K, V)

print(f'입력 토큰 수 L = {L}, 벡터 차원 d = {d}')
print('output shape:', out.shape, '  ← 각 토큰이 새 벡터로 갱신됨')
print('attention weights shape:', attn.shape, '  ← 각 행이 "해당 쿼리가 어느 키를 봤는지"')
print()
print('attention weights (각 행의 합은 1이어야 함):')
print(attn)
print('row sums:', attn.sum(axis=-1), '  ← softmax가 정상이면 모두 1.0')

### 1.1 Causal Mask (Decoder-only LLM의 핵심)

GPT계열 모델은 "**다음 토큰을 예측**"하는 학습을 한다. 학습 중에 한 문장 전체가 입력으로 들어가는데, 만약 아무 제한 없이 attention을 계산하면 **현재 토큰이 미래 토큰을 미리 본 뒤** "다음 단어"를 맞히는 일이 벌어진다. 그건 그냥 답지를 베끼는 셈.

그래서 **하삼각 마스크**로 미래 쪽을 가린다. 아래 표가 그 뜻이다. ✅는 볼 수 있고, 🚫는 못 본다.

| | 토큰0 | 토큰1 | 토큰2 | 토큰3 |
|---|---|---|---|---|
| **토큰0의 쿼리** | ✅ | 🚫 | 🚫 | 🚫 |
| **토큰1의 쿼리** | ✅ | ✅ | 🚫 | 🚫 |
| **토큰2의 쿼리** | ✅ | ✅ | ✅ | 🚫 |
| **토큰3의 쿼리** | ✅ | ✅ | ✅ | ✅ |

🚫 자리의 attention score에 $-\infty$(코드에서는 `-1e9`)를 더하면 softmax 후 값이 0이 된다 → 그 쪽은 참조하지 않는다.

In [4]:
def causal_mask(L):
    """하삼각 마스크: (L, L), 1 = 정보 전달 허용."""
    return np.tril(np.ones((L, L), dtype=np.int32))


mask = causal_mask(L)
print('causal mask:')
print(mask)

out_c, attn_c = scaled_dot_product_attention(Q, K, V, mask=mask)
print('\nmasked attention weights (상삼각이 0이 되어야 정상):')
print(attn_c)

causal mask:
[[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]

masked attention weights (상삼각이 0이 되어야 정상):
[[1.    0.    0.    0.   ]
 [0.105 0.895 0.    0.   ]
 [0.222 0.07  0.708 0.   ]
 [0.037 0.695 0.061 0.207]]


✅ 확인 포인트
- **토큰0**: 자기 자신만 볼 수 있으므로 가중치 100%가 자신에게 몰림
- **토큰1**: 토큰0, 토큰1 중에서 선택
- **토큰3**: 4개 모두 참조 가능

상삼각이 정확히 0이 되지 않고 아주 작은 값이 나올 수 있는데, 이는 `-1e9`가 완전한 $-\infty$가 아니기 때문이다 (실전에서는 충분히 무시할 수 있는 크기).

## 2. Multi-Head Attention (MHA)

### 왜 "여러 개의 head"가 필요한가?

지금까지 우리는 **attention을 1번** 계산했다. 하지만 실제 문장의 관계는 한 종류가 아니다:

> 예시: "**금감원**이 ○○**증권**에 **기관주의** 조치를 **결정**했다."

- 🔗 **주어-서술어** 관계: "금감원" ↔ "결정"
- 🔗 **주어-목적어** 관계: "금감원" ↔ "○○증권"
- 🔗 **동사-목적보어** 관계: "결정" ↔ "기관주의"

Head 하나로 이 관계들을 전부 담으려면 정보가 섞여 애매해진다. 그래서:

> 💡 $d_{model}$ 차원을 **head 수 $h$ 개로 쪼개** (각 $d_{head} = d_{model}/h$), 독립적으로 attention을 수행한 뒤 결과를 다시 붙인다. 각 head는 서로 다른 관계에 특화되도록 **자연스럽게 학습**된다.

### 수식
$$
\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O
\quad\text{where}\quad \text{head}_i = \text{Attention}(XW^Q_i, XW^K_i, XW^V_i)
$$

- 각 head는 자신만의 학습된 $W^Q_i, W^K_i, W^V_i$ 를 가진다.
- 마지막에 $W^O$ 로 다시 섞어 출력 차원을 맞춘다.

아래 `MultiHeadAttention` 클래스는 이 과정을 NumPy로 그대로 옮긴 **교육용 구현**이다. 학습 (gradient descent)은 하지 않고 난수 가중치만 쓴다.

In [5]:
class MultiHeadAttention:
    """교육용 최소 구현. 실전에서는 PyTorch/Flash-Attn을 쓴다."""

    def __init__(self, d_model=32, n_heads=4, seed=0):
        assert d_model % n_heads == 0, 'd_model은 n_heads의 배수여야 함'
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        rng = np.random.default_rng(seed)
        # 학습 가능한 선형변환을 난수로 초기화
        self.Wq = rng.standard_normal((d_model, d_model)) * 0.1
        self.Wk = rng.standard_normal((d_model, d_model)) * 0.1
        self.Wv = rng.standard_normal((d_model, d_model)) * 0.1
        self.Wo = rng.standard_normal((d_model, d_model)) * 0.1

    def _split_heads(self, X):
        # (L, d_model) -> (n_heads, L, d_head)
        L = X.shape[0]
        return X.reshape(L, self.n_heads, self.d_head).transpose(1, 0, 2)

    def _merge_heads(self, X):
        # (n_heads, L, d_head) -> (L, d_model)
        return X.transpose(1, 0, 2).reshape(-1, self.d_model)

    def __call__(self, X, mask=None):
        Q = self._split_heads(X @ self.Wq)
        K = self._split_heads(X @ self.Wk)
        V = self._split_heads(X @ self.Wv)

        head_outputs = []
        attn_per_head = []
        for h in range(self.n_heads):
            out_h, attn_h = scaled_dot_product_attention(Q[h], K[h], V[h], mask=mask)
            head_outputs.append(out_h)
            attn_per_head.append(attn_h)

        concat = self._merge_heads(np.stack(head_outputs))  # (L, d_model)
        return concat @ self.Wo, np.stack(attn_per_head)


L, d_model, n_heads = 6, 32, 4
X = np.random.randn(L, d_model)
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
out, attn_all = mha(X, mask=causal_mask(L))
print('output shape:', out.shape)
print('per-head attention shape:', attn_all.shape, '  # (n_heads, L, L)')

output shape: (6, 32)
per-head attention shape: (4, 6, 6)   # (n_heads, L, L)


### 2.1 MHA vs MQA vs GQA 비교

최근 LLM들은 **추론 속도**와 **메모리 절약**을 위해 K/V head 수를 줄인다. 이유는 4절의 KV Cache와 직결된다: Q는 매 스텝 1개 토큰만 계산하지만 **K/V는 과거 전체를 메모리에 들고 있어야** 하므로 K/V head 수가 곧 메모리 상한이 된다.

| 구조 | Q head 수 | K/V head 수 | KV Cache 크기 | 대표 모델 |
|---|---|---|---|---|
| **MHA** (Multi-Head Attention) | h | h | 100% | GPT-2, BERT |
| **MQA** (Multi-Query Attention) | h | **1** | 1/h | Falcon, PaLM |
| **GQA** (Grouped-Query Attention) | h | g (1 < g < h) | g/h | Llama2 70B, Llama3, Qwen2.5 |

### 🖼 한 장으로 기억하기

```
MHA :  Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8
       |  |  |  |  |  |  |  |
       K1 K2 K3 K4 K5 K6 K7 K8   ← 전부 1:1 (Q head 수 = K/V head 수)

MQA :  Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8
        \ \ \ \ / / / /
          ──── K1 ────           ← K/V head 1개를 8개의 Q가 공유

GQA :  Q1 Q2 Q3 Q4  Q5 Q6 Q7 Q8
        \  |  |  /   \  |  |  /
          K1               K2    ← 그룹 2개로 묶음 (g=2)
```

### 어떤 걸 쓸까?
- **MQA**: 가장 공격적 — 메모리는 가장 작지만 품질 저하가 올 수 있음.
- **GQA**: 중간 절충 — Llama3 70B는 64 Q / 8 KV head (g=8). 실전 LLM의 대세.
- **폐쇄망 배포 관점**: "같은 GPU 메모리에서 얼마나 **긴 context**를 담을 수 있는가"가 평가 기준. GQA/MQA가 유리. 모델 카드의 `num_key_value_heads` 필드로 확인 가능.

## 3. Positional Encoding — 순서 정보를 어떻게 넣을까?

### 왜 필요한가?
Self-Attention은 **순서 정보가 원래 없다**. 극단적인 실험으로 이해해 보자:

> 문장 A: "**고객**은 **은행**에 예금을 맡겼다." (고객이 맡김)
> 문장 B: "**은행**은 **고객**에 예금을 맡겼다." (은행이 맡김 — 의미 반전)

두 문장의 토큰 집합은 완전히 같다. Attention의 입력이 "토큰 벡터들의 집합"이라면, 두 문장이 **같은 출력**을 내놓는다 (이걸 "permutation invariant"라고 한다). 이러면 문법·의미를 학습할 수가 없다.

해결책: **위치에 따라 다른 벡터를 입력에 더해 주기.** 그러면 "은행@위치0"과 "은행@위치2"가 서로 다른 벡터가 되어 모델이 순서를 구분할 수 있다.

### 3.1 Sinusoidal PE — 원조 Transformer
$$
PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad
PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)
$$

- **왜 sin/cos?** — 서로 다른 주파수의 삼각함수 여러 개를 합치면 각 위치가 고유한 "지문"을 갖는다.
- **짝/홀 차원을 번갈아** 쓰는 이유: 두 위치의 차이가 **선형 변환**으로 표현 가능해 모델이 "상대 거리"를 학습하기 쉬움.
- **학습 파라미터 없음**: 공식으로 바로 계산.
- **장점**: 학습 때 안 본 길이까지 외삽 가능 (제한적).

### 3.1 Sinusoidal PE (원조 Transformer)

$$
PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)
$$

하나의 위치가 고유한 벡터로 매핑되고, 두 위치의 차이는 선형 변환으로 나타내질 수 있다.

In [ ]:
import matplotlib.pyplot as plt

from common import setup_korean_font
setup_korean_font()  # 한글 폰트가 있으면 자동 적용, 없으면 영문 그대로 표시


def sinusoidal_positional_encoding(max_len, d_model):
    """Sinusoidal PE 행렬 생성.

    Returns:
        pe: (max_len, d_model) — 각 행이 한 위치의 고유 벡터
    """
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, None]               # (max_len, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)            # 짝수 차원
    pe[:, 1::2] = np.cos(position * div_term)            # 홀수 차원
    return pe


pe = sinusoidal_positional_encoding(max_len=64, d_model=128)

# 왼쪽: 전체 히트맵, 오른쪽: 임의의 두 위치 벡터 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im = axes[0].imshow(pe, cmap='RdBu', aspect='auto')
axes[0].set_xlabel('dimension')
axes[0].set_ylabel('position')
axes[0].set_title('Sinusoidal PE (64 positions x 128 dims)')
plt.colorbar(im, ax=axes[0])

# 위치별 벡터 3개를 겹쳐 그려 "각 위치가 고유한 곡선"임을 시각화
for pos in [0, 10, 50]:
    axes[1].plot(pe[pos], label=f'pos={pos}', alpha=0.8)
axes[1].set_xlabel('dimension')
axes[1].set_ylabel('value')
axes[1].set_title('각 위치는 서로 다른 고유 벡터 (sin/cos 주파수 혼합)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 RoPE (Rotary Position Embedding) — Llama, Qwen, Mistral

핵심 아이디어를 한 줄로: **위치 정보를 덧셈이 아니라 "회전"으로 Q/K에 넣는다.**

차원을 2개씩 묶어 2D 평면 위의 점으로 보고, 각 위치 $m$ 에서 각도 $m\theta_i$ 만큼 시계 반대 방향으로 회전한다.

$$
\text{RoPE}(q, m)_i = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}
$$

**왜 좋은가?**
- $Q \cdot K^\top$ 계산 결과가 **두 위치의 상대 거리 $m-n$ 에만 의존**한다 → 문장 길이에 강건.
- 학습 때 2K, 추론 때 32K 같은 **길이 외삽**에 유리 (YaRN·NTK 보간 기법과 결합).
- V는 건드리지 않아 속도·메모리 영향 최소.

### 3.3 ALiBi (Attention with Linear Biases) — MPT, BLOOM

위치 임베딩 자체를 쓰지 않고, **attention score에 "거리 페널티"를 더하는** 아주 단순한 방법.

$$
\text{scores}_{ij} = \frac{Q_i K_j^\top}{\sqrt{d_k}} - m \cdot |i - j|
$$

- head 마다 고정된 기울기 $m$ 이 다르게 설정됨 → head별로 "가까운 쪽만 본다 / 멀리까지 본다"가 달라짐.
- 학습 파라미터 0개, 그래도 길이 외삽 성능이 훌륭함.

### 요약
| 방식 | 학습 파라미터 | Q/K/V 중 어디에 작용 | 상대 거리 표현 | 길이 외삽 |
|---|---|---|---|---|
| Sinusoidal | 0 | 입력에 덧셈 | 약함 | 제한적 |
| Learned | 있음 (위치 임베딩 테이블) | 입력에 덧셈 | 없음 | 약함 |
| **RoPE** | 0 | **Q, K에 회전** | **강함** | 좋음 |
| **ALiBi** | 0 | **score에 선형 페널티** | **강함** | 좋음 |

## 4. KV Cache — 왜 LLM 생성은 뒤로 갈수록 빠른가?

### 문제: 단순히 생성하면 $O(L^2)$
Decoder-only LLM의 생성(generation)은 **autoregressive** 하다. "안녕 하세요 오늘 은" 까지 만들었으면 다음 단어를 뽑기 위해 또 한 번 전체 문장을 모델에 넣는다. 매 스텝마다 전체 시퀀스의 Q, K, V를 다시 계산하면 **토큰 하나 늘어날 때마다 비용이 증가**해 총 비용이 $O(L^2)$.

### 관찰: 과거의 K, V는 바뀌지 않는다
모델 가중치는 고정이고, 과거 토큰들의 hidden state도 바뀌지 않으므로 **과거의 K, V를 저장해 재사용**할 수 있다. Q만 현재 토큰분 새로 계산하면 된다.

### 결과: 스텝당 $O(L)$ (누적 $O(L^2)$지만 상수 부담이 훨씬 작음)

```
스텝 t=0:  "전자"
  ├─ Q, K, V 모두 1개 토큰분 계산
  └─ KV Cache = [(K0, V0)]                        저장 크기: 1

스텝 t=1:  "금융"
  ├─ Q는 "금융" 1개만 계산
  ├─ K, V도 "금융" 1개만 계산 → 캐시에 이어붙임
  └─ KV Cache = [(K0,V0), (K1,V1)]                저장 크기: 2

스텝 t=2:  "거래"
  ├─ Q는 "거래" 1개만 계산
  ├─ K, V도 "거래" 1개만 계산 → 캐시에 이어붙임
  └─ KV Cache = [(K0,V0), (K1,V1), (K2,V2)]       저장 크기: 3
```

### 의사코드
```python
# (실제에는 per-layer, per-head로 저장)
kv_cache = {'K': None, 'V': None}
for t in range(max_new_tokens):
    q_t = project_q(hidden_t)          # (1, d)   현재 스텝만
    k_t = project_k(hidden_t)          # (1, d)
    v_t = project_v(hidden_t)          # (1, d)

    # 캐시에 이어붙인다
    kv_cache['K'] = concat(kv_cache['K'], k_t)  # (t+1, d)
    kv_cache['V'] = concat(kv_cache['V'], v_t)

    # 새 스텝 계산에는 q_t 와 전체 K/V만 필요
    out_t = softmax(q_t @ kv_cache['K'].T / sqrt(d)) @ kv_cache['V']
    next_token = sample(out_t)
```

### 메모리는 얼마나 드나? — Llama3-8B 기준
$$
\text{cache bytes} = 2 \times n_{\text{layers}} \times \text{seq\_len} \times n_{\text{kv\_heads}} \times d_{\text{head}} \times 2 \text{ bytes (fp16)}
$$
$= 2 \times 32 \times \text{seq\_len} \times 8 \times 128 \times 2 \approx$ **131 KB / 토큰**

| context 길이 | KV Cache 메모리 |
|---|---|
| 4K | 0.5 GB |
| 32K | 4 GB |
| 128K | 16 GB |

✅ **GQA가 없었다면** 이 값이 8배 (`n_kv_heads=64`) — 32K context에서만 32GB를 잡아먹어 일반 GPU에 못 들어감. 이게 MQA/GQA가 탄생한 이유다.

### 실무 팁
- 긴 context를 다루는 모델은 **KV Cache offloading**(CPU 메모리나 디스크로 이동), **paged attention** (vLLM) 같은 기법을 쓴다.
- **Speculative decoding / MTP**: 여러 토큰을 한꺼번에 검증 → 호출 횟수 절감. (Day 1 S1-5와 별개 주제)

## 5. 실전: 한국어 BERT로 attention 시각화

지금까지 NumPy로 직접 구현했다. 이제 **실제 사전 학습된 모델**이 한국어 문장을 처리할 때 어떤 attention 패턴을 보이는지 확인한다.

모델: `bert-base-multilingual-cased` (104개 언어 지원, 한국어 포함, 12 layer × 12 head = 144 attention 맵)

> 💡 BERT는 GPT와 달리 **양방향** (causal mask가 없다). 문장 이해에 최적화되어 attention 패턴이 비교적 선명하게 나타난다.

> ⚠️ 한글 폰트가 설치되지 않은 환경(일부 WSL/Linux)에서는 히트맵의 한글 틱 라벨이 □로 표시될 수 있다. 그 경우 아래 셀은 자동으로 `t0, t1, ...` 인덱스 라벨로 전환하고, 토큰 매핑을 텍스트로 출력한다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

sentence = '전자금융거래는 고객이 은행과 대면하지 않고 자동화된 방식으로 이용하는 거래입니다.'
inputs = tokenizer(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print('토큰 개수:', len(tokens))
print('토큰:', tokens)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

# attentions: tuple of (batch, n_heads, L, L) 길이 = n_layers
attentions = outputs.attentions
print('레이어 수:', len(attentions))
print('레이어 0 shape:', attentions[0].shape, '  # (batch=1, n_heads=12, L, L)')

In [ ]:
from common import safe_labels

# 특정 레이어·head 선택해 heatmap으로 시각화
LAYER = 6   # 중간 레이어
HEAD = 3    # 0~11

attn_matrix = attentions[LAYER][0, HEAD].numpy()  # (L, L)

# 한글 폰트가 있으면 그대로, 없으면 t0, t1, ... 인덱스 라벨로 대체
labels, legend = safe_labels(tokens)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(attn_matrix, cmap='viridis')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90)
ax.set_yticklabels(labels)
ax.set_title(f'Attention: layer={LAYER}, head={HEAD}')
ax.set_xlabel('Key (참조된 토큰)')
ax.set_ylabel('Query (참조하는 토큰)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# 한글 폰트가 없을 때는 t0→'전자'... 매핑을 텍스트로 출력해 그래프를 해석할 수 있게 함
if legend:
    print('[토큰 인덱스 매핑]')
    print(legend)

<!-- optional -->

### 5.1 여러 head를 한번에 비교

각 head가 서로 다른 역할을 학습했음을 확인한다.

In [ ]:
# <!-- optional -->
LAYER = 6
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for h, ax in enumerate(axes.flat):
    ax.imshow(attentions[LAYER][0, h].numpy(), cmap='viridis')
    ax.set_title(f'head {h}', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'Layer {LAYER} — 12 heads')
plt.tight_layout()
plt.show()


## 더 읽어보기
- Vaswani et al., [Attention Is All You Need (2017)](https://arxiv.org/abs/1706.03762)
- Su et al., [RoFormer: Enhanced Transformer with Rotary Position Embedding (2021)](https://arxiv.org/abs/2104.09864)
- Ainslie et al., [GQA: Training Generalized Multi-Query Transformer Models (2023)](https://arxiv.org/abs/2305.13245)
- [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)